In [ ]:
import os
import datetime
import geopandas as gpd
import multiprocessing as mp
import pandas as pd
import sentinelhub

import cdseutils.utils
import cdseutils.sentinel2

In [2]:
CDSE_JSON_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/cdse_credentials.json'

TRAINING_DATA_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops_sel_region_2018_sampled.geojson'
INFERENCE_FILEPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_test_download_geometry.geojson'

ID_COL = 'Name' # column name in the ROI_FILEPATH which uniquely identifies each polygon
START_DATE_STR = '2018-01-01'
END_DATE_STR = '2018-12-31'

SENTINEL2_LEVEL = 'l2a'
BANDS = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B11', 'B12', 'SCL']
MOSAIC_DAYS = 20
SCL_MASK_CLASSES = [0, 1, 3, 7, 8, 9, 10]

NJOBS = mp.cpu_count() - 2


# TO DO: hide the paths below + put the three functions into one call
TILE_DOWNLOAD_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/satellite/sentinel-2-l2a'
MPC_TILE_DOWNLOAD_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/satellite_mpc/sentinel-2-l2a'
SNAKEMAKE_PATH = '../snakemake/create_datacube_inmemory/Snakefile'
DATACUBE_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/datacubes'
MPC_DATACUBE_FOLDERPATH = '/Users/nikhilsrajan/NASA-Harvest/project/fetch_satdata/data/at_eurocrops/datacubes_mpc'

In [3]:
shapes_gdf = gpd.read_file(INFERENCE_FILEPATH)

In [4]:
CDSE_CREDS = cdseutils.utils.cdse_credentials_from_json(CDSE_JSON_FILEPATH)

START_DATE = datetime.datetime.strptime(START_DATE_STR, '%Y-%m-%d')
END_DATE = datetime.datetime.strptime(END_DATE_STR, '%Y-%m-%d')

SENTINEL2_CATALOG_FILEPATH = os.path.join(TILE_DOWNLOAD_FOLDERPATH, 'catalog_sentinel-2.geojson')
MPC_SENTINEL2_CATALOG_FILEPATH = os.path.join(MPC_TILE_DOWNLOAD_FOLDERPATH, 'catalog.geojson')

TIMESTAMP_COL = 'timestamp' # column name in the SENTINEL2_CATALOG_FILEPATH which tells the datetime of image acquisition

DATACUBE_CATALOG_FILEPATH = os.path.join(DATACUBE_FOLDERPATH, 'input.csv')
MPC_DATACUBE_CATALOG_FILEPATH = os.path.join(MPC_DATACUBE_FOLDERPATH, 'input.csv')

In [5]:
collection = sentinelhub.DataCollection.SENTINEL2_L2A
satellite = 'sentinel-2-l2a'

In [ ]:
catalog_gdf = cdseutils.utils.query_catalog(
    shapes_gdf = shapes_gdf,
    sh_creds = CDSE_CREDS.sh_creds,
    collection = collection,
    startdate = START_DATE,
    enddate = END_DATE,
    cache_folderpath = None,
)

In [ ]:
catalog_gdf['s3url']

In [ ]:
s3paths, download_filepaths = \
cdseutils.sentinel2.get_s3paths(
    s3urls = catalog_gdf['s3url'],
    s3_creds = CDSE_CREDS.s3_creds,
    root_folderpath = TILE_DOWNLOAD_FOLDERPATH,
    bands = BANDS,
    satellite = satellite,
)

In [ ]:
s3paths

In [ ]:
download_filepaths